# 01 · Multi-Layer Perceptron (MLP) with a sliding window

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LekshmiSR/proteins-d1-hands-on/blob/main/notebooks/01_mlp_sliding_window.ipynb)

**Idea.** An MLP (a stack of fully connected layers) needs a **fixed-size input vector**. A protein has a variable length, so we cannot feed the whole protein. Instead, to classify residue *i* we feed the MLP a **window** of residues around it: positions *i−7 … i+7* (15 residues). We slide this window along the chain, one residue at a time. This is the classic approach of early secondary-structure predictors (e.g. PSIPRED in the 1990s used windows of 15).

```
sequence:   ... K  T  L  N  E  G [L] G  L  L  E  R  V ...
                 \___________ window of 15 ___________/
                              |
                     flatten into one vector
                              |
        Linear → ReLU → Linear → ReLU → Linear → 3 scores (C, H, E) for the centre residue [L]
```

This notebook is the most detailed one: besides the model, it also walks through **one training step** by hand (forward pass, loss, gradients, update). Later notebooks reuse the same training code.

Expected time on a Colab T4: about 3–5 minutes.

## 0. Setup

Run the next cell first. It works in two places:

* **Google Colab**: it downloads this repository (code + helpers) into the Colab machine.
* **Your own computer** (Jupyter / VS Code): it just moves to the repository folder.

> **Colab tip:** use a GPU. Menu **Runtime → Change runtime type → T4 GPU → Save**. Do this *before* running cells.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LekshmiSR/proteins-d1-hands-on.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if "google.colab" in sys.modules:                       # running in Colab
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from notebooks/
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protein_dl import data, train, plots

train.set_seed(42)
device = train.get_device()

## 1. Settings

All the knobs in one place. Change them and re-run the notebook to experiment.

* `TRAIN_SUBSET = 2000` uses only 2000 training proteins: handy for a quick try on a CPU.

In [ ]:
WINDOW = 15          # residues seen by the MLP (must be odd: centre residue + 7 on each side)
EMB_DIM = 32         # size of the learned vector for each amino acid
HIDDEN = (256, 128)  # sizes of the two hidden layers
DROPOUT = 0.2

EPOCHS = 15
BATCH_SIZE = 32
LR = 1e-3            # learning rate
MAX_LEN = 512        # training proteins longer than this are randomly cropped
TRAIN_SUBSET = None  # e.g. 2000 for a quick test; None = all 17,799 proteins
SAVE_TO_DRIVE = False

## 2. Load the data

The same loaders as in notebook 00 (see there for every detail). We grab one batch to push through the model step by step.

In [ ]:
train_dl, valid_dl, test_dl = data.make_dataloaders("data", batch_size=BATCH_SIZE, max_len=MAX_LEN,
                                                     train_subset=TRAIN_SUBSET)
tokens, labels, lengths = next(iter(train_dl))
print("tokens :", tokens.shape, "  <- [B, L]")
print("labels :", labels.shape)
print("lengths:", lengths.shape)

## 3. Build the model step by step

We follow one batch through every operation and print the tensor shape after each step.
`B` = batch size, `L` = length of the longest protein in the batch, `E` = embedding size, `W` = window.

### Step 1: embedding
Each token becomes a learnable vector of size `EMB_DIM`. `padding_idx=0` keeps the padding vector fixed at zero.

In [ ]:
embedding = nn.Embedding(data.VOCAB_SIZE, EMB_DIM, padding_idx=data.PAD_IDX)
x = embedding(tokens)
print("tokens   :", tuple(tokens.shape), "  [B, L]")
print("embedded :", tuple(x.shape), "  [B, L, E]")

### Step 2: cut a window around every residue

Residues at the start and end of a protein do not have 7 neighbours on one side, so we first pad 7 extra `PAD` tokens on both ends.
Then `unfold` cuts out all windows of length 15, with step 1. Result: one window per residue.

In [ ]:
half = WINDOW // 2
padded = F.pad(tokens, (half, half), value=data.PAD_IDX)
print("padded tokens :", tuple(padded.shape), f"  [B, L + {2 * half}]")

xe = embedding(padded)                       # [B, L+14, E]
windows = xe.unfold(dimension=1, size=WINDOW, step=1)
print("windows       :", tuple(windows.shape), "  [B, L, E, W]  (unfold puts the window last)")

windows = windows.transpose(2, 3)            # [B, L, W, E]  (residue order first, nicer to read)
flat = windows.flatten(start_dim=2)          # [B, L, W*E]
print("flattened     :", tuple(flat.shape), f"  [B, L, W*E] = [B, L, {WINDOW}*{EMB_DIM}]")

Let us check that the window of residue 0 really is "7 pads + the first 8 residues":

In [ ]:
win_tokens = padded.unfold(1, WINDOW, 1)     # same operation on the raw tokens
print("first 10 tokens of protein 0  :", tokens[0, :10].tolist())
print("window of residue 0           :", win_tokens[0, 0].tolist())
print("window of residue 1           :", win_tokens[0, 1].tolist())

### Step 3: the MLP itself

Now every residue is described by one vector of length `W*E = 480`. The MLP is applied to each of these vectors independently (PyTorch `Linear` layers work on the last dimension, so `[B, L, 480]` is fine).

* `Linear(in, out)`: `y = x @ Wᵀ + b`, every output neuron is connected to every input number.
* `ReLU`: sets negative numbers to 0. Without such a non-linearity, stacked linear layers would collapse into one linear layer.
* `Dropout`: during training, randomly zeroes 20% of the numbers. This reduces overfitting.
* The last `Linear` outputs **3 numbers** per residue: the logits for C, H, E.

In [ ]:
mlp = nn.Sequential(
    nn.Linear(WINDOW * EMB_DIM, HIDDEN[0]), nn.ReLU(), nn.Dropout(DROPOUT),
    nn.Linear(HIDDEN[0], HIDDEN[1]),        nn.ReLU(), nn.Dropout(DROPOUT),
    nn.Linear(HIDDEN[1], data.NUM_CLASSES),
)
h = flat
for layer in mlp:
    h = layer(h)
    print(f"{layer.__class__.__name__:8s} -> {tuple(h.shape)}")
print("\nlogits:", tuple(h.shape), "  [B, L, 3]  one score per class for every residue")

### Put it together as a model class

A PyTorch model is a class with two parts: `__init__` creates the layers (with their weights), `forward` says how data flows through them.

In [ ]:
class WindowMLP(nn.Module):
    def __init__(self, window=WINDOW, emb_dim=EMB_DIM, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.window = window
        self.embedding = nn.Embedding(data.VOCAB_SIZE, emb_dim, padding_idx=data.PAD_IDX)
        self.mlp = nn.Sequential(
            nn.Linear(window * emb_dim, hidden[0]), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden[0], hidden[1]),        nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden[1], data.NUM_CLASSES),
        )

    def forward(self, tokens, lengths=None):          # lengths is not needed by this model
        half = self.window // 2
        x = self.embedding(F.pad(tokens, (half, half), value=data.PAD_IDX))  # [B, L+2h, E]
        x = x.unfold(1, self.window, 1)                                     # [B, L, E, W]
        x = x.transpose(2, 3).flatten(2)                                    # [B, L, W*E]
        return self.mlp(x)                                                  # [B, L, 3]


model = WindowMLP().to(device)
print(model)
print(f"\nTrainable parameters: {train.count_parameters(model):,}")

Where do the parameters come from? Each `Linear(in, out)` has `in*out` weights + `out` biases:

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:20s} {str(tuple(p.shape)):12s} {p.numel():>8,}")

## 4. One training step, by hand

Training repeats four steps on batch after batch:

1. **Forward**: compute the logits for the batch.
2. **Loss**: measure how wrong they are. We use **cross-entropy**: for each residue, `−log(probability given to the true class)`. Padding positions (label `−100`) are ignored.
3. **Backward**: `loss.backward()` computes the **gradient** of the loss with respect to every weight (how the loss would change if the weight moved a tiny bit).
4. **Update**: the optimizer moves every weight a small step against its gradient.

Before training, the model is guessing. A random 3-class guess gives a loss of about `ln(3) ≈ 1.10`.

In [ ]:
tokens_d, labels_d = tokens.to(device), labels.to(device)

logits = model(tokens_d, lengths)
loss = train.masked_cross_entropy(logits, labels_d)
print("logits:", tuple(logits.shape))
print(f"loss before training: {loss.item():.4f}   (ln 3 = {np.log(3):.4f})")

probs = logits[0, :5].softmax(dim=-1)
print("\nprobabilities for the first 5 residues of protein 0 (C, H, E):")
print(probs.detach().cpu().numpy().round(3))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

optimizer.zero_grad()          # clear old gradients
loss.backward()                # compute new gradients
w = model.mlp[0].weight
print("first layer weight:", tuple(w.shape), "  its gradient:", tuple(w.grad.shape))
before = w.detach().clone()
optimizer.step()               # update the weights
print(f"average change of first-layer weights: {(w.detach() - before).abs().mean().item():.2e}")

# Do a few more steps on the same batch: the loss on it should drop.
for step in range(1, 21):
    optimizer.zero_grad()
    loss = train.masked_cross_entropy(model(tokens_d, lengths), labels_d)
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f"step {step:2d}: loss on this batch = {loss.item():.4f}")

The loss drops: the model is learning (here only from one batch, which is cheating a little). Real training loops over **all** batches (one pass = one **epoch**), and checks the **validation** set after every epoch.

This is the full loop we use, from `protein_dl/train.py` (we added mixed precision on GPU and gradient clipping for stability):

In [ ]:
import inspect
print(inspect.getsource(train.train_one_epoch))

## 5. Train for real

We start again from a fresh model so that the hand-made steps above do not count.

In [ ]:
train.set_seed(42)
model = WindowMLP().to(device)
history = train.train_model(model, train_dl, valid_dl, device, epochs=EPOCHS, lr=LR)

In [ ]:
plots.plot_history(history, "Window MLP")

**Reading the curves:** training loss should go down steadily. If validation loss starts going *up* while training loss keeps going down, the model is **overfitting** (memorising the training proteins). `train_model` keeps the weights of the best validation epoch, which protects us from that.

## Evaluate on the test set (CB513)

The model now has the weights from its best validation epoch. We run it **once** on CB513, the independent test set.

In [ ]:
val_metrics = train.evaluate(model, valid_dl, device)
test_metrics = train.evaluate(model, test_dl, device)

print(f"Validation Q3: {val_metrics['q3_accuracy']:.4f}")
print(f"CB513 Q3     : {test_metrics['q3_accuracy']:.4f}   macro F1: {test_metrics['macro_f1']:.4f}")
print()
print("Per-class scores on CB513:")
for cls, s in test_metrics["per_class"].items():
    print(f"  {cls:6s} precision {s['precision']:.3f}  recall {s['recall']:.3f}  F1 {s['f1']:.3f}  ({s['support']} residues)")

In [ ]:
plots.plot_confusion(test_metrics["confusion_matrix"], "Window MLP on CB513 (rows sum to 1)")

**How to read the confusion matrix:** each row is a true class and shows where its residues ended up. The diagonal is the recall of each class. Off-diagonal cells show the typical mistakes (strands predicted as coil is usually the biggest one).

Let us look at a real prediction for one CB513 protein:

In [ ]:
trues, preds, _ = train.predict(model, test_dl, device)
i = 100                                           # try other numbers!
x = np.array(test_dl.dataset.tokens[i]); y = np.array(test_dl.dataset.labels[i])
seq = "".join(data.AMINO_ACIDS[t - 1] if 1 <= t <= 20 else "X" for t in x[y != data.IGNORE_INDEX])
print(f"protein {i}: {len(seq)} scored residues, accuracy {(trues[i] == preds[i]).mean():.2f}\n")
plots.show_prediction(seq, trues[i], preds[i])

## Save the results

The results (metrics, training curves, settings) are written to `results/mlp.json`. Notebook 05 reads these files to compare all models.

In Colab, files disappear when the session ends. Set `SAVE_TO_DRIVE = True` in the settings cell at the top to also keep a copy in your Google Drive (Colab will ask for permission).

In [ ]:
drive_dir = train.get_drive_dir(SAVE_TO_DRIVE)
config = dict(EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, MAX_LEN=MAX_LEN, TRAIN_SUBSET=TRAIN_SUBSET)
results = train.save_results("mlp", model, history, val_metrics, test_metrics, config=config, drive_dir=drive_dir)

## What did we learn?

* The MLP clearly beats the amino-acid propensity baseline from notebook 00 (≈ 49%): **context matters**.
* Its weakness: it only ever sees 15 residues, and it treats "residue at position −3" and "residue at position −2" as completely unrelated inputs, with separate weights. It has to learn the same pattern again for every position in the window.

The **CNN** in the next notebook fixes the second point by sliding one small shared filter along the chain.

## Exercises

1. Set `WINDOW = 1` (no context at all). What Q3 do you get? Compare with the propensity baseline.
2. Try `WINDOW = 7`, `21`, `31`. Does a bigger window always help?
3. Remove the `Dropout` layers (set `DROPOUT = 0`). What happens to the gap between training and validation loss?